# EDA — Retail Banking Customer Segmentation

100,000 synthetic retail-banking customers. Not ICICI data.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
df=pd.read_csv(Path("../data/customer_data_clean.csv"))
print(df.shape)

## 1. Inspect the first five rows

In [ ]:
df.head()

## 2. Column names and types

In [ ]:
df.info()

## 3. Missing values

In [ ]:
m=df.isna().sum().sort_values(ascending=False)
print(m[m>0])
print("Total missing cells:",int(m.sum()))

## 4. Duplicate checks

In [ ]:
print("Duplicate rows:",df.duplicated().sum())
print("Duplicate IDs:",df.customer_id.duplicated().sum())

## 5. Descriptive statistics and skewness

In [ ]:
num=df.select_dtypes(include=np.number)
st=num.describe().T
st["skewness"]=num.skew()
st.loc[["annual_income","account_balance","monthly_transaction_count","monthly_transaction_amount","credit_limit","credit_utilization_ratio","loan_amount","investment_amount"]].round(2)

## 6. Categorical distributions

In [ ]:
for c in ["gender","city_tier","occupation","card_category"]:
    print("\n",c); print(df[c].value_counts())

## 7. Domain validation

In [ ]:
checks={"Age":df.age.between(18,100).all(),"Income":(df.annual_income>0).all(),"Balance":(df.account_balance>=0).all(),"Utilization":df.credit_utilization_ratio.between(0,1).all(),"Digital":df.digital_transaction_ratio.between(0,1).all(),"UPI":df.upi_transaction_share.between(0,1).all()}
checks

## 8. Income distribution

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(np.log1p(df.annual_income),bins=50)
plt.title("Annual Income — log1p")
plt.xlabel("log1p(Annual Income)"); plt.ylabel("Customers"); plt.show()

## 9. Account balance distribution

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(np.log1p(df.account_balance),bins=50)
plt.title("Account Balance — log1p")
plt.xlabel("log1p(Account Balance)"); plt.ylabel("Customers"); plt.show()

## 10. Transaction activity

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df.monthly_transaction_count,bins=50)
plt.title("Monthly Transaction Count")
plt.xlabel("Transactions/Month"); plt.ylabel("Customers"); plt.show()

## 11. Digital usage

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df.digital_transaction_ratio,bins=40)
plt.title("Digital Transaction Ratio")
plt.xlabel("Digital Ratio"); plt.ylabel("Customers"); plt.show()

## 12. Credit utilization

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df.credit_utilization_ratio,bins=40)
plt.title("Credit Utilization Ratio")
plt.xlabel("Utilization"); plt.ylabel("Customers"); plt.show()

## 13. City tier mix

In [ ]:
x=df.city_tier.value_counts().sort_index()
plt.figure(figsize=(8,5)); plt.bar(x.index,x.values); plt.title("Customers by City Tier"); plt.xlabel("City Tier"); plt.ylabel("Customers"); plt.show()

## 14. Income vs balance

In [ ]:
s=df.sample(10000,random_state=42)
plt.figure(figsize=(8,5)); plt.scatter(np.log1p(s.annual_income),np.log1p(s.account_balance),s=8,alpha=.35); plt.title("Income vs Account Balance — 10k Sample"); plt.xlabel("log1p(Income)"); plt.ylabel("log1p(Balance)"); plt.show()

## 15. Correlation matrix

In [ ]:
cols=["annual_income","account_balance","monthly_transaction_count","monthly_transaction_amount","credit_card_spend","credit_limit","credit_utilization_ratio","loan_amount","investment_amount","digital_transaction_ratio","months_inactive_12_mon"]
c=df[cols].corr()
plt.figure(figsize=(10,8)); plt.imshow(c.values,aspect="auto"); plt.xticks(range(len(cols)),cols,rotation=80,ha="right",fontsize=8); plt.yticks(range(len(cols)),cols,fontsize=8); plt.title("Correlation Matrix"); plt.colorbar(); plt.tight_layout(); plt.show()

## EDA conclusion

The financial and behavioral variables show useful variation and skew. This motivates business-index feature engineering before K-Means.